In [93]:
import pandas as pd
import numpy as np
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [94]:
# 연체일을 나타내는 delay_days 피처 생성
train['clear_date'] = pd.to_datetime(train['clear_date'])
train['due_in_date'] = pd.to_datetime(train['due_in_date'])
train['delay_days'] = (train['clear_date'] - train['due_in_date']).dt.days

test['clear_date'] = pd.to_datetime(test['clear_date'])
test['due_in_date'] = pd.to_datetime(test['due_in_date'])
test['delay_days'] = (test['clear_date'] - test['due_in_date']).dt.days

In [ ]:
bins = [-float('inf'), 0, 5, 15, float('inf')]
labels = ['on_time_or_early', '1_5', '6_15', '16_plus']

# 4개의 클래스로 타깃 피처 생성(정상, 1-5일 지연, 6-15일 지연, 16이상 지연)
train['delay_class'] = pd.cut(train['delay_days'], bins=bins, labels=labels)
train['delay_class'].value_counts(dropna=False)
train['delay_class'].value_counts(normalize=True, dropna=False) * 100

test['delay_class'] = pd.cut(test['delay_days'], bins=bins, labels=labels)
test['delay_class'].value_counts(dropna=False)
test['delay_class'].value_counts(normalize=True, dropna=False) * 100

delay_class
on_time_or_early    60.8500
1_5                 31.5000
6_15                 4.9125
16_plus              2.7375
Name: proportion, dtype: float64

In [ ]:
# 타깃 피처와 직접적으로 연관되어있거나(clear_date, delay_days) 아무런 정보도 없는 열(posting_id, isOpen, business_year, area_business), 단순 식별자(doc_id, invoice_id), ,현재는 사용하기 힘든 열들(cust_number, name_customer, 날짜관련 변수들)
drop_cols = ['target', 'clear_date', 'delay_days', 'posting_id', 'isOpen','buisness_year','area_business','doc_id', 'invoice_id', 'document_create_date', 'document_create_date.1', 'due_in_date', 'baseline_create_date','posting_date', 'cust_number', 'name_customer']
train.drop(columns=drop_cols, axis=1, inplace=True)
test.drop(columns=drop_cols, axis=1, inplace=True)


In [85]:
train = train.dropna().copy()
test = test.dropna().copy()

In [86]:
from sklearn.preprocessing import OneHotEncoder
# 타깃 피처 분리
train_y = train['delay_class'].copy()
train_x = train.drop(columns='delay_class', axis=1)
test_y = test['delay_class'].copy()
test_x = test.drop(columns='delay_class', axis=1)

#원핫 인코딩
cat_cols = train_x.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = train_x.select_dtypes(exclude=['object', 'category']).columns.tolist()
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
train_cat = ohe.fit_transform(train_x[cat_cols])
test_cat = ohe.transform(test_x[cat_cols])

train_cat_df = pd.DataFrame(
    train_cat,
    columns=ohe.get_feature_names_out(cat_cols),
    index=train_x.index
)

test_cat_df = pd.DataFrame(
    test_cat,
    columns=ohe.get_feature_names_out(cat_cols),
    index=test_x.index
)

train_x = pd.concat([train_x[num_cols], train_cat_df], axis=1)
test_x = pd.concat([test_x[num_cols], test_cat_df], axis=1)


In [97]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
lr = LogisticRegression(
    max_iter=5000,
    random_state=42,
    class_weight='balanced'
)

lr.fit(train_x, train_y)
pred = lr.predict(test_x)

/home/wagyu0923/miniconda3/envs/invoice/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [98]:
labels = ['on_time_or_early', '1_5', '6_15', '16_plus']
print('accuracy:', accuracy_score(test_y, pred))
print('-------------')
print('macro f1:', f1_score(test_y, pred, average='macro'))
print('-------------')
print(classification_report(test_y, pred, labels = labels))
print('-------------')
print(confusion_matrix(test_y, pred, labels=labels))

accuracy: 0.545625
-------------
macro f1: 0.4742484548143595
-------------
                  precision    recall  f1-score   support

on_time_or_early       0.78      0.53      0.63      4868
             1_5       0.41      0.58      0.48      2520
            6_15       0.21      0.43      0.29       393
         16_plus       0.44      0.57      0.50       219

        accuracy                           0.55      8000
       macro avg       0.46      0.53      0.47      8000
    weighted avg       0.63      0.55      0.56      8000

-------------
[[2600 1940  247   81]
 [ 686 1472  322   40]
 [  45  143  169   36]
 [  10   32   53  124]]
